In [1]:
from tensorflow.keras.datasets import imdb

# Keep only top 10,000 most frequent words
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


#Step 2: Preprocessing – Padding the Sequences


All reviews need to be the same length for neural networks.

In [2]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 200  # We'll pad or truncate reviews to 200 words
X_train_padded = pad_sequences(X_train, maxlen=maxlen)
X_test_padded = pad_sequences(X_test, maxlen=maxlen)


#Step 3: Embedding Layer
Instead of raw word indices, we use word embeddings to give dense vector representation for each word.

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

embedding_dim = 128

embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


What Is an Embedding Layer?


An embedding layer takes each word index (integer) in your input sequence and maps it to a dense vector (i.e., a vector of real numbers). This helps the model understand the meaning and context of words.


 Why Not Just Use Word Indexes :
 ------------------------------

Suppose your vocabulary looks like this:

Word	      Index
"amazing"  	1
"terrible"	2
"movie"	    3
"great"	    4
If you pass these numbers directly to a neural network:

The model may think word "2" is twice as meaningful as word "1" — which is not true.

Integers have no semantic meaning.

So instead, we convert each word index into a vector that captures meaning — like how "amazing" and "great" should be close in vector space.





In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

embedding_dim = 128

embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


input_dim=vocab_size:
--------------------
Total number of unique tokens (words) in your vocabulary.
Example: If you have 10,000 unique words → vocab_size = 10000.


output_dim=embedding_dim:
-------------------------
This is the size of the vector each word index will be converted into.
Typical sizes are 50, 100, 128, 300. Higher values = more expressive but slower.

input_length=maxlen:
------------------------
This is the length of each input sequence after padding.
Example: If all your sentences are padded to 200 words → maxlen = 200.


Resulting Output Shape:
-------------------------

If your input is a batch of shape:
(batch_size, maxlen) → e.g. (32, 200)

After the embedding layer, the shape becomes:
(batch_size, maxlen, embedding_dim) → e.g. (32, 200, 128)

So each word is now represented by a 128-dim vector, not just an integer.



#RNN

In [8]:
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_model = Sequential([
    embedding_layer,
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
rnn_model.fit(X_train_padded, y_train, epochs=3, batch_size=64, validation_data=(X_test_padded, y_test))


Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 53s 126ms/step - accuracy: 0.5784 - loss: 0.6597 - val_accuracy: 0.6788 - val_loss: 0.5905
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 80s 123ms/step - accuracy: 0.8073 - loss: 0.4393 - val_accuracy: 0.7957 - val_loss: 0.4734
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 82s 123ms/step - accuracy: 0.9084 - loss: 0.2297 - val_accuracy: 0.6973 - val_loss: 0.6486


validation_data:
----------------

validation_data is a separate set of data (not used for training) that the model evaluates after each epoch to check how well it's learning on unseen data.


Why is it Useful:
--------------------

1. Monitors Overfitting
It helps you check if the model is memorizing training data or actually learning patterns.



If training accuracy keeps increasing but validation accuracy drops → 🚨 Overfitting

If both training and validation improve → 👍 Good learning

2. Helps Early Stopping
You can set a callback like EarlyStopping to automatically stop training if the validation loss stops improving.

3. Debugging
You can catch issues like:

Model being too simple (underfitting)

Not enough data

Incorrect preprocessing (e.g., wrong tokenization or padding)

#LSTM

In [10]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=maxlen),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()
lstm_model.fit(X_train_padded, y_train, epochs=3, batch_size=64, validation_data=(X_test_padded, y_test))



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 156s 391ms/step - accuracy: 0.6850 - loss: 0.5657 - val_accuracy: 0.8630 - val_loss: 0.3230
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 201s 388ms/step - accuracy: 0.8992 - loss: 0.2607 - val_accuracy: 0.8710 - val_loss: 0.3229
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 201s 385ms/step - accuracy: 0.9341 - loss: 0.1747 - val_accuracy: 0.8678 - val_loss: 0.3511


#GRU

In [11]:
from tensorflow.keras.layers import GRU

gru_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=maxlen),
    GRU(64),
    Dense(1, activation='sigmoid')
])

gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
gru_model.summary()
gru_model.fit(X_train_padded, y_train, epochs=3, batch_size=64, validation_data=(X_test_padded, y_test))


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 163s 408ms/step - accuracy: 0.7141 - loss: 0.5170 - val_accuracy: 0.8610 - val_loss: 0.3264
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 203s 410ms/step - accuracy: 0.9035 - loss: 0.2458 - val_accuracy: 0.8760 - val_loss: 0.3065
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 149s 381ms/step - accuracy: 0.9377 - loss: 0.1732 - val_accuracy: 0.8680 - val_loss: 0.3279


#Compare Model Performance

In [12]:
rnn_score = rnn_model.evaluate(X_test_padded, y_test)
lstm_score = lstm_model.evaluate(X_test_padded, y_test)
gru_score = gru_model.evaluate(X_test_padded, y_test)

print(f"RNN Accuracy: {rnn_score[1]:.4f}")
print(f"LSTM Accuracy: {lstm_score[1]:.4f}")
print(f"GRU Accuracy: {gru_score[1]:.4f}")


782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.6896 - loss: 0.6630
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 40ms/step - accuracy: 0.8680 - loss: 0.3574
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 39ms/step - accuracy: 0.8682 - loss: 0.3288
RNN Accuracy: 0.6973
LSTM Accuracy: 0.8678
GRU Accuracy: 0.8680


#Important points to remember while using or creating LSTM RNN GRU models

# Common Steps Before Implementing RNN / LSTM / GRU
These steps are your standard pre-processing pipeline for sequence models.


 Step 1: Understand the Problem Type
 ------------------------------------

Is it classification (e.g., positive/negative)?

Or regression (e.g., predicting stock price)?

Or sequence-to-sequence (e.g., translation)?

This affects the model's final layer and loss function.


Step 2: Collect and Clean the Data
------------------------------------

For text data: remove noise, HTML tags, symbols, lowercase, etc.

For time series: remove outliers, fill missing values.


Step 3: Tokenization / Feature Extraction
-----------------------------------------

Convert input into sequences of numbers.


 1. Tokenization
 -----------------

When? → When your input is text (e.g., reviews, tweets, articles).
Why? → Neural networks can’t read words — they need numbers.

What it does:
-----------------

Converts text into a sequence of integers, where each number represents a unique word.


text = ["I love dogs", "I hate cats"]

After tokenization:
------------------

"I love dogs" → [1, 2, 3]
"I hate cats" → [1, 4, 5]

Each word is mapped to an index.


from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(text_data)

sequences = tokenizer.texts_to_sequences(text_data)


Step 4: Padding / Sequence Formatting
--------------------------------------

Make all sequences the same length (required for batching).

 Padding / Truncating Sequences
 -------------------------------

When? → When your sequences (text or time series) are not all the same length.

Why? → Neural networks (RNN/LSTM/GRU) require inputs to be uniform in size.

What it does:
-------------

Makes all sequences the same length by:

Padding (adding 0s) to short sequences

Truncating long sequences

[[1, 2, 3], [4, 5]]  → pad to maxlen=3
→ [[1, 2, 3],
    [0, 4, 5]]


-----

from tensorflow.keras.preprocessing.sequence import pad_sequences



padded = pad_sequences(sequences, maxlen=200)



Step 5: Define Vocabulary Size and Sequence Length
------------------------------------------------

Needed for embedding layers:
----------------------------

Vocabulary Size (num_words or vocab_size)
----------------------------------------

When? → During tokenization.

Why? → To limit the number of words the model learns (helps with memory and performance).


What it does:
--------------

Keeps only the top n most frequent words.

Example:
--------

Tokenizer(num_words=5000)  → Only top 5000 frequent words are kept

All rare words are ignored or mapped to a special “OOV” (out of vocabulary) token.

Step 6: Split into Train / Test (and optionally Validation)
------------------------------------------------------------



from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(padded, labels, test_size=0.2)

 Step 7: Create Embedding Layer
 ------------------------------

Used only for text (not for time series usually)



from tensorflow.keras.layers import Embedding

embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen)


Step 8: Define Model Architecture
------------------------------------

Use RNN / LSTM / GRU
----------------------


from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense

model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),
    LSTM(64),  # or SimpleRNN or GRU
    Dense(1, activation='sigmoid')  # or 'softmax' for multi-class
])

Step 9: Compile and Train
------------------------

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=64)

 Step 10: Evaluate and Compare
 ------------------------------

 model.evaluate(X_test, y_test)




